<a href="https://colab.research.google.com/github/nmatthews5/GithubAction/blob/master/Masters.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

# Load the datasets
perf_df = pd.read_excel('/content/golf_player_dataset_full.xlsx')
odds_df = pd.read_excel('/content/golf_finishing_position_odds.xlsx')

print("Performance Data Sample:")
display(perf_df.head())
print("\nBetting Odds Sample:")
display(odds_df.head())

Performance Data Sample:


,Rank,Player,Fracas,Cut %,1st Rd Win %,Win %,Top 20 %,Top 10 %,Top 5 %
0,1,Jon Rahm,2.805,94.6,6.2,8.5,76.6,54.8,35.7
1,2,Matt Fitzpatrick,2.675,93.8,6.1,7.6,74.4,52.0,33.2
2,3,Ludvig Aberg,2.441,91.2,5.9,6.7,68.0,46.0,28.8
3,4,Bryson DeChambeau,2.444,91.0,5.7,6.0,67.1,44.8,27.5
4,5,Cameron Young,2.322,90.3,5.4,5.5,64.8,42.3,25.7



Betting Odds Sample:


,Player,Winner,Top 5,Top 10,Top 20,Top 30,Top 40
0,Scottie Scheffler,550,138,-137,-345,-670,-2000
1,Bryson DeChambeau,1000,250,120,-200,-500,-1000
2,Jon Rahm,1000,250,120,-200,-500,-1000
3,Rory McIlroy,1200,300,125,-175,-335,-835
4,Ludvig Åberg,1400,335,150,-175,-295,-715


### Betting Math Logic
To determine if a bet is worth it, we use this logic:
1. **Convert American Odds to Decimal Odds**:
   - If Odds (+) : `(Odds / 100) + 1`
   - If Odds (-) : `(100 / abs(Odds)) + 1`
2. **Implied Probability**: `1 / Decimal Odds`
3. **Value Identification**: If `Your Percentage > Implied Probability`, it is a positive value bet.

In [13]:
def american_to_implied(odds):
    if odds > 0:
        return 100 / (odds + 100)
    else:
        return abs(odds) / (abs(odds) + 100)

# We will need to merge these dataframes on the Player Name once we confirm column names.
print("Awaiting data inspection to perform the merge and EV calculation.")

Awaiting data inspection to perform the merge and EV calculation.


In [18]:
import unicodedata

def normalize_name(name):
    """Remove accents and lowercase for better matching."""
    if not isinstance(name, str): return name
    # Normalize unicode characters (like ø to o)
    name = unicodedata.normalize('NFKD', name).encode('ascii', 'ignore').decode('utf-8')
    return name.strip().lower()

# 1. Normalize Player names
perf_df['Player_norm'] = perf_df['Player'].apply(normalize_name)
odds_df['Player_norm'] = odds_df['Player'].apply(normalize_name)

# Manual Overrides for known name variations
name_map = {
    'matthew fitzpatrick': 'matt fitzpatrick',
    'nicolai hojgaard': 'nicolai hjgaard' # Matching the current odd string to perf string
}
odds_df['Player_norm'] = odds_df['Player_norm'].replace(name_map)

# 2. Merge dataframes
merged_df = pd.merge(perf_df, odds_df, on='Player_norm', suffixes=('_perf', '_odds'))

# 3. Define all categories
categories = {
    'Winner': 'Win %',
    'Top 5': 'Top 5 %',
    'Top 10': 'Top 10 %',
    'Top 20': 'Top 20 %'
}

results = []
for _, row in merged_df.iterrows():
    res = {'Player': row['Player_odds']}
    for odd_col, perf_col in categories.items():
        if odd_col in row and perf_col in row:
            implied = american_to_implied(row[odd_col]) * 100
            actual = row[perf_col]
            edge = actual - implied
            res[f'{odd_col}_Edge'] = round(edge, 2)
            res[f'{odd_col}_Value'] = "YES" if edge > 0 else "NO"
    results.append(res)

value_df = pd.DataFrame(results)

print(f"Successfully calculated odds")

Successfully calculated odds


In [10]:
print("Full Winner Value Bets Table:")
# Displaying the Player, Win Edge, and the Value decision
winner_value_df = value_df[['Player', 'Winner_Edge', 'Winner_Value']].sort_values(by='Winner_Edge', ascending=False)
display(winner_value_df)

Full Winner Value Bets Table:


,Player,Winner_Edge,Winner_Value
1,Matthew Fitzpatrick,2.84,YES
5,Collin Morikawa,2.26,YES
7,Robert MacIntyre,1.45,YES
11,Akshay Bhatia,0.81,YES
12,Jake Knapp,0.54,YES
4,Cameron Young,0.24,YES
16,J.J. Spaun,0.16,YES
2,Ludvig Åberg,0.03,YES
13,Si Woo Kim,-0.04,NO
17,Russell Henley,-0.46,NO


In [14]:
print("Full Top 10 Value Bets Table:")
# Displaying the Player, Top 10 Edge, and the Value decision
top10_value_df = value_df[['Player', 'Top 10_Edge', 'Top 10_Value']].sort_values(by='Top 10_Edge', ascending=False)
display(top10_value_df)

Full Top 10 Value Bets Table:


,Player,Top 10_Edge,Top 10_Value
5,Collin Morikawa,19.60,YES
1,Matthew Fitzpatrick,18.67,YES
7,Robert MacIntyre,12.43,YES
11,Akshay Bhatia,9.62,YES
0,Jon Rahm,9.35,YES
12,Jake Knapp,6.80,YES
16,J.J. Spaun,6.22,YES
2,Ludvig Åberg,6.00,YES
4,Cameron Young,5.94,YES
13,Si Woo Kim,5.61,YES


In [15]:
print("Full Top 20 Value Bets Table:")
# Displaying the Player, Top 20 Edge, and the Value decision
top20_value_df = value_df[['Player', 'Top 20_Edge', 'Top 20_Value']].sort_values(by='Top 20_Edge', ascending=False)
display(top20_value_df)

Full Top 20 Value Bets Table:


,Player,Top 20_Edge,Top 20_Value
5,Collin Morikawa,24.16,YES
1,Matthew Fitzpatrick,18.84,YES
7,Robert MacIntyre,14.30,YES
11,Akshay Bhatia,13.14,YES
16,J.J. Spaun,10.69,YES
14,Patrick Reed,10.36,YES
12,Jake Knapp,10.08,YES
0,Jon Rahm,9.93,YES
13,Si Woo Kim,9.28,YES
4,Cameron Young,6.99,YES


In [11]:
# Select relevant columns from both original dataframes to display side-by-side
# We use the merged_df created in the previous step

display_cols = [
    'Player_odds', 'Win %', 'Winner',
    'Top 5 %', 'Top 5',
    'Top 10 %', 'Top 10',
    'Top 20 %', 'Top 20'
]

# Filter to columns that exist (in case some percentages were missing)
existing_display_cols = [c for c in display_cols if c in merged_df.columns]

master_comparison_df = merged_df[existing_display_cols].rename(columns={'Player_odds': 'Player Name'})

print(f"Displaying {len(master_comparison_df)} players found in both datasets:")
display(master_comparison_df.sort_values(by='Win %', ascending=False))

Displaying 26 players found in both datasets:


,Player Name,Win %,Winner,Top 5 %,Top 5,Top 10 %,Top 10,Top 20 %,Top 20
0,Jon Rahm,8.5,1000,35.7,250,54.8,120,76.6,-200
1,Matthew Fitzpatrick,7.6,2000,33.2,450,52.0,200,74.4,-125
2,Ludvig Åberg,6.7,1400,28.8,335,46.0,150,68.0,-175
3,Bryson DeChambeau,6.0,1000,27.5,250,44.8,120,67.1,-200
4,Cameron Young,5.5,1800,25.7,450,42.3,175,64.8,-137
5,Collin Morikawa,5.2,3300,26.3,600,44.6,300,68.6,125
6,Xander Schauffele,4.9,1400,24.6,335,41.9,160,66.0,-162
7,Robert MacIntyre,4.9,2800,24.0,550,41.0,250,64.3,100
8,Scottie Scheffler,4.8,550,23.5,138,39.8,-137,62.4,-345
9,Tommy Fleetwood,2.9,2000,17.5,450,32.8,200,56.3,-125


In [16]:
# Identify the Edge columns
edge_cols = [c for c in value_df.columns if '_Edge' in c]

# Create a comparison table with the Player name and all edge values
edge_comparison_df = value_df[['Player'] + edge_cols].copy()

# Calculate an 'Average Edge' to help identify the top 5 overall value players
edge_comparison_df['Average_Edge'] = edge_comparison_df[edge_cols].mean(axis=1)

# Sort by Average Edge and display the Top 5
print("Comparison of Edge Values Across All Categories (Top 5 Overall Value Players):")
display(edge_comparison_df.sort_values(by='Average_Edge', ascending=False).head(5))

Comparison of Edge Values Across All Categories (Top 5 Overall Value Players):


,Player,Winner_Edge,Top 5_Edge,Top 10_Edge,Top 20_Edge,Average_Edge
5,Collin Morikawa,2.26,12.01,19.60,24.16,14.5075
1,Matthew Fitzpatrick,2.84,15.02,18.67,18.84,13.8425
7,Robert MacIntyre,1.45,8.62,12.43,14.30,9.2000
11,Akshay Bhatia,0.81,4.70,9.62,13.14,7.0675
0,Jon Rahm,-0.59,7.13,9.35,9.93,6.4550
